In [ ]:
# 在 Google Colab 里运行时，先把本书的代码拉下来（本地 Jupyter 会跳过这一格）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/pytorch-inside-out"):
        !git clone -q --depth 1 https://github.com/wikieden/pytorch-inside-out /content/pytorch-inside-out
    %cd /content/pytorch-inside-out/code/notebooks


> 在线讲义：[第 20 章 · DeviceMesh、DTensor 与张量并行](https://wikieden.github.io/pytorch-inside-out/lessons/20-dtensor-tp.html) · 本书《PyTorch 从里到外》

# 第 20 章 · DeviceMesh、DTensor 与张量并行

本章回答：一个张量"分布在多张卡上"怎么描述？DTensor 的 Shard / Replicate / Partial 是什么意思？张量并行（TP）怎么把一个 MLP 切到两张卡上，为什么"先按列切、再按行切"只要通信一次？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/20-dtensor-tp.md`（讲解、动画、练习都在讲义里）。对应原文：Ultra-Scale Playbook §Tensor Parallelism（`ultrascale-playbook.md` 第 1032 行起：TP in a Transformer Block 第 1107 行、Sequence Parallelism 第 1214 行）；Megatron-LM 论文（〔在线〕arXiv:1909.08053）§3；官方文档 DTensor、`torch.distributed.tensor.parallel`（〔在线〕）；多机报告 §2.2；本章输出在本机 Gloo 2 进程上跑过

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import sys
sys.path.insert(0, "..")   # 子进程要能按名字导入函数，所以用本地模块

from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.tensor import Replicate, Shard, distribute_tensor
from torch.distributed.tensor.parallel import ColwiseParallel, RowwiseParallel, parallelize_module
import torch
import torch.nn as nn

import ch20_dtensor_tp as M          # 传给 run(...) 的函数从这里取
from dist_util import run
torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 Shard(0) | 按行切 | `dtensor_demo` 的 local | [[0, 1], [2, 3]] / [[4, 5], [6, 7]] |
| 第二步 Partial | sum 后部分和 | 同上 | Partial(sum)，full_tensor 56 |
| 第三步 TP | W1 列切、W2 行切 | `tp_demo` | TP 与不切都是 [[2, 4]]；W1 Shard(0)、W2 Shard(1) |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `dtensor_demo`**（来自 `ch20_dtensor_tp.py`）：下面要用到的小工具。

In [ ]:
def dtensor_demo(rank, world):
    mesh = init_device_mesh("cpu", (world,))
    dt = distribute_tensor(torch.arange(8.0).reshape(4, 2), mesh, [Shard(0)])
    rep = dt.redistribute(mesh, [Replicate()])
    s = (dt * 2).sum()
    return dict(local=dt.to_local().tolist(), replicate_shape=tuple(rep.to_local().shape),
                sum_placement=str(s.placements), sum=s.full_tensor().item())

**类 `MLP`**（来自 `ch20_dtensor_tp.py`）：下面要用到的小工具。

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.w1 = nn.Linear(2, 4, bias=False)
        self.w2 = nn.Linear(4, 2, bias=False)

    def forward(self, x):
        return self.w2(torch.relu(self.w1(x)))

**函数 `tp_demo`**（来自 `ch20_dtensor_tp.py`）：下面要用到的小工具。

In [ ]:
def tp_demo(rank, world):
    mesh = init_device_mesh("cpu", (world,))
    m = MLP()
    with torch.no_grad():
        m.w1.weight.copy_(torch.tensor([[1.0, 0], [0, 1], [1, 0], [0, 1]]))     # (out=4, in=2)：就是讲义 W1 的转置
        m.w2.weight.copy_(torch.tensor([[1.0, 0, 1, 0], [0, 1, 0, 1]]))         # (out=2, in=4)
    x = torch.tensor([[1.0, 2.0]])
    full = m(x).tolist()
    parallelize_module(m, mesh, {"w1": ColwiseParallel(), "w2": RowwiseParallel()})
    return dict(full=full, tp=m(x).tolist(), w1=(tuple(m.w1.weight.to_local().shape), str(m.w1.weight.placements)),
                w2=(tuple(m.w2.weight.to_local().shape), str(m.w2.weight.placements)))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 DTensor：")
for r, res in enumerate(run(M.dtensor_demo)):
    print(f"  rank {r}: {res}")

In [ ]:
print("\n第三步 MLP 张量并行：")
for r, res in enumerate(run(M.tp_demo)):
    print(f"  rank {r}: {res}")

## 3. 代码里才看得到的

1. **代码里的权重是讲义矩阵的转置**：`nn.Linear` 存 (out, in)，所以"按列切"显示成 Shard(0)。
2. **`full_tensor()` 也是集合通信**：两个 rank 都要调用（第 22 章的坑）。

## 4. 自己改一改

- 把 `tp_demo` 的 x 改成 [[3, 1]]，先手算再跑。
- `dt.redistribute(mesh, [Shard(1)])` 看每卡拿到什么。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> mesh 把卡排成表，每一维一种并行一个组；
> DTensor 三种摆：Shard 切开、Replicate 复制、Partial 部分和，换摆法就是通信；
> TP 先列后行，中间不传、最后一次 all_reduce；只在机内用，SP 再把 norm 按序列切。